In [3]:
# -----------------------------
# 1. Imports
# -----------------------------
import os, random, time
from typing import List
from dataclasses import dataclass

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score

# -----------------------------
# 2. Reproducibility
# -----------------------------
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)

# -----------------------------
# 3. Dataset
# -----------------------------
class FERVA(Dataset):
    """Facial Expression + Valence/Arousal Dataset"""
    def __init__(self, images_dir, ann_dir, ids: List[str], transform=None):
        self.images_dir = images_dir
        self.ann_dir = ann_dir
        self.ids = ids
        self.transform = transform

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        img_id = self.ids[idx]
        img_path = os.path.join(self.images_dir, f"{img_id}.jpg")
        img = Image.open(img_path).convert("RGB")
        exp = int(np.load(os.path.join(self.ann_dir, f"{img_id}_exp.npy")).item())
        val = float(np.load(os.path.join(self.ann_dir, f"{img_id}_val.npy")))
        aro = float(np.load(os.path.join(self.ann_dir, f"{img_id}_aro.npy")))

        if self.transform:
            img = self.transform(img)

        return {"image": img, "exp": exp, "val": val, "aro": aro}

# -----------------------------
# 4. Multi-task model
# -----------------------------
class MultiOutputModel(nn.Module):
    def __init__(self, backbone="resnet18", num_classes=8):
        super().__init__()
        if backbone=="resnet18":
            self.backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
            in_features = self.backbone.fc.in_features
            self.backbone.fc = nn.Identity()
        elif backbone=="efficientnet_b0":
            self.backbone = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
            in_features = self.backbone.classifier[1].in_features
            self.backbone.classifier = nn.Identity()
        else:
            raise ValueError("Unsupported backbone")

        # Classification head
        self.head_cls = nn.Sequential(
            nn.Linear(in_features, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
        # Valence head
        self.head_val = nn.Sequential(
            nn.Linear(in_features, 128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 1), nn.Tanh()
        )
        # Arousal head
        self.head_aro = nn.Sequential(
            nn.Linear(in_features, 128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 1), nn.Tanh()
        )

    def forward(self, x):
        feats = self.backbone(x)
        logits = self.head_cls(feats)
        val = self.head_val(feats).squeeze(1)
        aro = self.head_aro(feats).squeeze(1)
        return logits, val, aro

# -----------------------------
# 5. Metrics
# -----------------------------
def classification_metrics(y_true, y_prob):
    y_pred = y_prob.argmax(axis=1)
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "F1_Macro": f1_score(y_true, y_pred, average="macro"),
        "CohenKappa": cohen_kappa_score(y_true, y_pred)
    }

def regression_metrics(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    rmse = np.sqrt(np.mean((y_true-y_pred)**2))
    corr = np.corrcoef(y_true, y_pred)[0,1] if len(y_true) > 1 else 0.0
    return {"RMSE": rmse, "CORR": corr}

# -----------------------------
# 6. Training utilities
# -----------------------------
@dataclass
class Config:
    backbone: str
    lr: float = 1e-4
    weight_decay: float = 1e-4
    epochs: int = 20
    lambdas: tuple = (1.0,0.5,0.5)

def train_one_epoch(model, loader, optimizer=None, device="cuda", loss_weights=(1.0,0.5,0.5)):
    ce = nn.CrossEntropyLoss()
    mse = nn.MSELoss()
    model.train() if optimizer else model.eval()
    total_loss = 0.0

    y_true_cls, y_prob_cls = [], []
    y_true_val, y_pred_val = [], []
    y_true_aro, y_pred_aro = [], []

    for batch in loader:
        x = batch["image"].to(device)
        y_cls = torch.tensor(batch["exp"], dtype=torch.long, device=device)
        y_val = torch.tensor(batch["val"], dtype=torch.float32, device=device)
        y_aro = torch.tensor(batch["aro"], dtype=torch.float32, device=device)

        logits, val, aro = model(x)
        loss = loss_weights[0]*ce(logits, y_cls) + loss_weights[1]*mse(val, y_val) + loss_weights[2]*mse(aro, y_aro)

        if optimizer:
            optimizer.zero_grad(); loss.backward(); optimizer.step()

        total_loss += loss.item() * x.size(0)
        y_true_cls.extend(y_cls.detach().cpu().numpy())
        y_prob_cls.append(torch.softmax(logits,1).detach().cpu().numpy())
        y_true_val.extend(y_val.detach().cpu().numpy()); y_pred_val.extend(val.detach().cpu().numpy())
        y_true_aro.extend(y_aro.detach().cpu().numpy()); y_pred_aro.extend(aro.detach().cpu().numpy())

    y_prob_cls = np.vstack(y_prob_cls)
    avg_loss = total_loss / max(1,len(loader.dataset))
    return avg_loss, classification_metrics(y_true_cls, y_prob_cls), regression_metrics(y_true_val, y_pred_val), regression_metrics(y_true_aro, y_pred_aro)

def train_model(cfg: Config, train_loader, val_loader, device="cuda"):
    model = MultiOutputModel(cfg.backbone).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    best_val_loss = float('inf')

    for ep in range(cfg.epochs):
        tr_loss, _, _, _ = train_one_epoch(model, train_loader, optimizer=optimizer, device=device, loss_weights=cfg.lambdas)
        val_loss, cls_m, val_m, aro_m = train_one_epoch(model, val_loader, optimizer=None, device=device, loss_weights=cfg.lambdas)
        print(f"[{cfg.backbone}] Epoch {ep+1}/{cfg.epochs} TrainLoss={tr_loss:.4f} ValLoss={val_loss:.4f} Acc={cls_m['Accuracy']:.3f}")
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), f"best_{cfg.backbone}.pt")
    return model

# -----------------------------
# 7. Dataset & DataLoaders
# -----------------------------
dataset_path =  "/kaggle/input/dataset/Dataset"
images_dir = os.path.join(dataset_path, "images")
ann_dir = os.path.join(dataset_path, "annotations")

all_ids = sorted([f.split(".")[0] for f in os.listdir(images_dir) if f.endswith(".jpg")])
labels = [int(np.load(os.path.join(ann_dir, f"{i}_exp.npy"))) for i in all_ids]

# Stratified split: 70% train, 15% val, 15% test
sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, temp_idx = next(sss1.split(all_ids, labels))
temp_ids = [all_ids[i] for i in temp_idx]; temp_labels = [labels[i] for i in temp_idx]
sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
val_idx, test_idx = next(sss2.split(temp_ids, temp_labels))
train_ids = [all_ids[i] for i in train_idx]
val_ids = [temp_ids[i] for i in val_idx]
test_ids = [temp_ids[i] for i in test_idx]

train_tfms = T.Compose([
    T.Resize((224,224)), T.RandomHorizontalFlip(0.5),
    T.ColorJitter(0.2,0.2,0.2,0.05),
    T.RandomAffine(degrees=10, translate=(0.05,0.05), scale=(0.95,1.05)),
    T.ToTensor(), T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
eval_tfms = T.Compose([T.Resize((224,224)), T.ToTensor(), T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

train_ds = FERVA(images_dir, ann_dir, train_ids, transform=train_tfms)
val_ds   = FERVA(images_dir, ann_dir, val_ids, transform=eval_tfms)
test_ds  = FERVA(images_dir, ann_dir, test_ids, transform=eval_tfms)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

# -----------------------------
# 8. Train & Evaluate
# -----------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"

for backbone in ["resnet18", "efficientnet_b0"]:
    print(f"\n=== TRAINING {backbone.upper()} ===")
    cfg = Config(backbone=backbone, epochs=20)
    model = train_model(cfg, train_loader, val_loader, device=device)

    model.load_state_dict(torch.load(f"best_{backbone}.pt"))
    _, cls_m, val_m, aro_m = train_one_epoch(model, test_loader, optimizer=None, device=device)

    print(f"\n=== {backbone.upper()} TEST METRICS ===")
    print("Classification:", cls_m)
    print("Valence Metrics:", val_m)
    print("Arousal Metrics:", aro_m)



=== TRAINING RESNET18 ===


/tmp/ipykernel_36/164344026.py:139: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_cls = torch.tensor(batch["exp"], dtype=torch.long, device=device)
/tmp/ipykernel_36/164344026.py:140: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_val = torch.tensor(batch["val"], dtype=torch.float32, device=device)
/tmp/ipykernel_36/164344026.py:141: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_aro = torch.tensor(batch["aro"], dtype=torch.float32, device=device)
/tmp/ipykernel_36/164344026.py:139: UserWarning: To copy construct from a tensor, it is 

[resnet18] Epoch 1/20 TrainLoss=2.1121 ValLoss=1.8454 Acc=0.322


/tmp/ipykernel_36/164344026.py:139: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_cls = torch.tensor(batch["exp"], dtype=torch.long, device=device)
/tmp/ipykernel_36/164344026.py:140: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_val = torch.tensor(batch["val"], dtype=torch.float32, device=device)
/tmp/ipykernel_36/164344026.py:141: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_aro = torch.tensor(batch["aro"], dtype=torch.float32, device=device)
/tmp/ipykernel_36/164344026.py:139: UserWarning: To copy construct from a tensor, it is 

[resnet18] Epoch 2/20 TrainLoss=1.7437 ValLoss=1.7416 Acc=0.367


...

[efficientnet_b0] Epoch 19/20 TrainLoss=0.2431 ValLoss=2.5318 Acc=0.422


/tmp/ipykernel_36/164344026.py:139: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_cls = torch.tensor(batch["exp"], dtype=torch.long, device=device)
/tmp/ipykernel_36/164344026.py:140: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_val = torch.tensor(batch["val"], dtype=torch.float32, device=device)
/tmp/ipykernel_36/164344026.py:141: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_aro = torch.tensor(batch["aro"], dtype=torch.float32, device=device)
/tmp/ipykernel_36/164344026.py:139: UserWarning: To copy construct from a tensor, it is 

[efficientnet_b0] Epoch 20/20 TrainLoss=0.2339 ValLoss=2.5786 Acc=0.423


/tmp/ipykernel_36/164344026.py:139: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_cls = torch.tensor(batch["exp"], dtype=torch.long, device=device)
/tmp/ipykernel_36/164344026.py:140: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_val = torch.tensor(batch["val"], dtype=torch.float32, device=device)
/tmp/ipykernel_36/164344026.py:141: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_aro = torch.tensor(batch["aro"], dtype=torch.float32, device=device)



=== EFFICIENTNET_B0 TEST METRICS ===
Classification: {'Accuracy': 0.465, 'F1_Macro': 0.4606314242747569, 'CohenKappa': 0.38857142857142857}
Valence Metrics: {'RMSE': 0.39676112, 'CORR': 0.5653890340022435}
Arousal Metrics: {'RMSE': 0.3487403, 'CORR': 0.4493016881843371}
